# Inference and Bootstraps

A fit model gives more than estimates: each coefficient comes with a *standard error*, measuring how much its estimate would vary from one dataset to the next. From the standard errors come tests and confidence intervals. This notebook covers:

- where `py-glm`'s standard errors come from, and the tests and intervals built on them,
- what a "95% confidence interval" promises, checked by simulation,
- estimating standard errors a second way, by resampling, with the bootstrap, and
- what goes wrong when a model's assumption about the variance of the response is wrong, and how the bootstrap reveals it.

As in the other notebooks, every simulation has its own seed.

In [1]:
import numpy as np
import pandas as pd

from glm.datasets import make_frame
from glm.families import Poisson, QuasiPoisson
from glm.glm import GLM
from glm.simulation import Simulation, coefficient_table

## Standard errors, tests, and intervals

Here are counts from a Poisson model with a few predictors, two of them correlated, and a categorical variable:

In [2]:
sim = make_frame(
    Poisson(),
    n_samples=1000,
    n_uncorrelated=2,
    n_correlated=2,
    correlation=0.5,
    categorical={"region": 3},
    coef_range=(-0.4, 0.4),
    rng=np.random.default_rng(1),
)
model = GLM(Poisson()).fit(sim.data, formula=sim.formula)

`summary_frame` collects each coefficient's estimate, standard error, test statistic, p-value, and confidence interval:

In [3]:
model.summary_frame().round(4)

,estimate,std error,z,p-value,lower 95%,upper 95%
Intercept,0.4717,0.0424,11.1230,0.0000,0.3886,0.5549
x1,0.0386,0.0258,1.4939,0.1352,-0.0120,0.0892
x2,0.2939,0.0251,11.7223,0.0000,0.2448,0.3431
x3,0.2156,0.0296,7.2951,0.0000,0.1577,0.2736
x4,0.0258,0.0290,0.8889,0.3740,-0.0310,0.0826
C(region)[T.b],-0.0879,0.0613,-1.4340,0.1516,-0.2081,0.0322
C(region)[T.c],-0.2780,0.0631,-4.4081,0.0000,-0.4016,-0.1544


Where do the standard errors come from? Fitting a GLM [maximizes the log-likelihood](https://en.wikipedia.org/wiki/Maximum_likelihood_estimation). For each of our exponential families, this has a closed form. For the Poisson family, its:

```
f(μ | y) = μ - y log(μ)
μ = exp(x @ β)
```

summed over the observed data `(x, y)`.

The curvature of the log-likelihood at its maximum is the *information matrix*, it measures how sharply the data pin down the coefficients. Its inverse, scaled by the (estimated) dispersion `φ`, estimates the covariance matrix of the coefficient estimates.  The reported standard errors are the square roots of its diagonal. 

The justification for this procedure is a theorem from likelihood theory which concludes that, asymptotically, estimates produced by maximum likelihood [converge in distribution to a multi-variate normal](https://gregorygundersen.com/blog/2019/11/28/asymptotic-normality-mle/), with covariance matrix exactly as we computed.

Recall, for the Poisson family, `φ` is fixed at 1:

In [4]:
covariance = model.dispersion_ * np.linalg.inv(model.information_matrix_)
np.allclose(np.sqrt(np.diag(covariance)), model.coef_standard_error_)

True

The test statistic is `estimate / std error`, testing the hypothesis that a true coefficient is precisely zero. It's compared with the standard normal distribution here, so it's labelled `z`; families that estimate their dispersion, like the Gaussian, compare it with Student's `t` distribution instead, accounting for the uncertainty in the dispersion, and label it `t`. Likewise, a 95% confidence interval is `estimate ± 1.96 × std error`, the 1.96 the standard normal's 97.5th percentile:

In [5]:
frame = model.summary_frame()
np.allclose(frame["lower 95%"], frame["estimate"] - 1.96 * frame["std error"], atol=1e-3)

True

## What "95%" means

A 95% confidence interval comes with a promise about *the procedure*, not any one interval: if we collected many datasets from the same population, and computed an interval from each, about 95% of those intervals would contain the true coefficient. With simulated data, where we know the truth, we can check that promise directly. This function simulates many datasets from the same model, fits each one, and records how often each coefficient's interval contains its true value:

In [6]:
def coverage(family, n_datasets=500, level=0.95, latent_noise_sd=0.0, **simulation):
    """How often each coefficient's confidence interval, fit with the
    family, contains the true coefficient, over many simulated datasets."""
    covered = []
    for seed in range(n_datasets):
        sim = make_frame(
            Poisson(), latent_noise_sd=latent_noise_sd, rng=np.random.default_rng(seed), **simulation
        )
        model = GLM(family).fit(sim.data, formula=sim.formula, offset=sim.offset)
        frame = model.summary_frame(level=level)
        # Latent noise (any noise added to the linear predictor that is invisible to the features) 
        # raises the average count. A model that can't see the noise attributes the effect to its 
        # intercept; see below.
        truth = sim.coef.copy()
        truth["Intercept"] += latent_noise_sd**2 / 2
        percent = f"{100 * level:g}%"
        covered.append(
            (frame[f"lower {percent}"] <= truth) & (truth <= frame[f"upper {percent}"])
        )
    return pd.DataFrame(covered).mean().rename("coverage")

In [7]:
simulation_params = dict(
    n_samples=1000,
    n_uncorrelated=2,
    n_correlated=2,
    correlation=0.5,
    categorical={"region": 3},
    coef_range=(-0.4, 0.4),
)

poisson_coverage = coverage(Poisson(), **simulation_params)
poisson_coverage.round(3)

Intercept         0.956
x1                0.946
x2                0.944
x3                0.962
x4                0.964
C(region)[T.b]    0.962
C(region)[T.c]    0.968
Name: coverage, dtype: float64

Every coefficient's interval covers its true value close to 95% of the time. With 500 datasets, a coverage estimate has a standard error of about `√(0.95 × 0.05 / 500) ≈ 0.01`, so values within about two percentage points of 95% are what a correct procedure gives:

In [8]:
print(f"Average coverage: {poisson_coverage.mean():.3f}")

Average coverage: 0.957


## Bootstraps

The bootstrap estimates standard errors a different way: refit the model to many datasets resembling the original, and measure how much the estimates vary. There are two ways to make those datasets.

- The **parametric bootstrap** simulates new responses from the *fit model*, for the same predictors.
- The **non-parametric bootstrap** resamples rows of the original data, with replacement.

Each returns a list of fit models, and `coefficient_table` collects their estimates, one row per bootstrap replicate:

In [9]:
bootstrap = Simulation(model)
np.random.seed(2)
parametric = coefficient_table(bootstrap.parametric_bootstrap(sim.data, n_sim=500))
non_parametric = coefficient_table(bootstrap.non_parametric_bootstrap(sim.data, n_sim=500))

parametric.head()

,Intercept,x1,x2,x3,x4,C(region)[T.b],C(region)[T.c]
0,0.412754,0.067788,0.329126,0.216595,0.010257,-0.065111,-0.183161
1,0.458828,0.035613,0.276316,0.221597,0.000323,-0.058182,-0.281521
2,0.431786,0.026289,0.316335,0.201916,0.035763,-0.056089,-0.181627
3,0.457171,0.025313,0.284269,0.229006,0.049099,-0.069038,-0.245023
4,0.407110,0.024962,0.306334,0.205351,0.002281,-0.032704,-0.190773


To repeat a little for emphasis: each row in this (parametric bootstrap) table encapsulates *resampling* data by using the fit model to determine `y | X` for each observation, then sampling a new value of `y` for each row in the original data. This procedure produces a new dataset, with a different, randomized, `y`. We then refit the same model to this new data, get new coefficients, and those new coefficients make up a single row of this table.

The standard deviation of each coefficient's estimates across replicates is the bootstrap standard error. In this case, the model is specified correctly (since our original dataset was simulated from the exact structure as the model). The three estimates agree, to within the noise in each: 

 - The parametric bootstrap uses a finite number of replicates, so there is estimation error. As the number of replicates -> ∞ we expect convergence.
 -  The non-parametric bootstrap's standard errors carry extra noise of their own, they are observed to be a little larger on average.  We'll see more about this below.

In [10]:
pd.DataFrame(
    {
        "model": model.coef_standard_error_,
        "parametric bootstrap": parametric.std(),
        "non-parametric bootstrap": non_parametric.std(),
    },
    index=model.X_names,
).round(4)

,model,parametric bootstrap,non-parametric bootstrap
Intercept,0.0424,0.0409,0.0445
x1,0.0258,0.0254,0.0281
x2,0.0251,0.0249,0.0277
x3,0.0296,0.0288,0.0304
x4,0.0290,0.0288,0.0279
C(region)[T.b],0.0613,0.0580,0.0632
C(region)[T.c],0.0631,0.0640,0.0661


The bootstrap also gives confidence intervals directly: the middle 95% of the replicates' estimates, the *percentile interval*. They agree well with the model's intervals:

In [11]:
percentile = non_parametric.quantile([0.025, 0.975]).T
percentile.columns = ["lower 95%", "upper 95%"]
pd.concat(
    {"model": frame[["lower 95%", "upper 95%"]], "non-parametric bootstrap": percentile},
    axis=1,
).round(3)

model           non-parametric bootstrap          
               lower 95% upper 95%                lower 95% upper 95%
Intercept          0.389     0.555                    0.380     0.550
x1                -0.012     0.089                   -0.017     0.091
x2                 0.245     0.343                    0.242     0.350
x3                 0.158     0.274                    0.156     0.277
x4                -0.031     0.083                   -0.030     0.076
C(region)[T.b]    -0.208     0.032                   -0.211     0.052
C(region)[T.c]    -0.402    -0.154                   -0.403    -0.153

## When the variance assumption is wrong

A Poisson model assumes the variance of each count equals its mean, `Var[y | X] = E[y | X]`. Counts collected from observed phenomena in the real world are often more variable than that, for example because of predictors we didn't measure. This is called *overdispersion*.

Our `make_frame`'s `latent_noise_sd` allows simulating exactly that, adding unobserved normal noise to the linear predictor. That is, we simulate:

```
    y | X ~ Poisson(λ = exp(X @ β + ϵ))
    ϵ ~ Normal(m = 0, σ = 0.5)
```

With σ = 0.5, the quantity `σ² / 2 = 0.125` will come up a few times below.

In [12]:
overdispersed = dict(
    **simulation_params, 
    # The exposures raise the counts, which keeps this example clear 
    # of small counts complications.
    exposure_range=(5.0, 20.0),
    # Adds unexplainable noise to the linear predictor.
    latent_noise_sd=0.5
)

sim = make_frame(Poisson(), rng=np.random.default_rng(3), **overdispersed)
poisson_model = GLM(Poisson()).fit(sim.data, formula=sim.formula, offset=sim.offset)
quasi_model = GLM(QuasiPoisson()).fit(sim.data, formula=sim.formula, offset=sim.offset)

print(f"Estimated dispersion: {quasi_model.dispersion_:.2f}")

Estimated dispersion: 6.12


How do the estimates from the two models differ, and how do they compare to the truth? 

In [13]:
pd.concat([
    pd.Series(poisson_model.coef_, index=poisson_model.X_names).rename('poisson'),
    pd.Series(quasi_model.coef_, index=quasi_model.X_names).rename('quasi-poisson'),
    pd.Series(sim.coef, index=poisson_model.X_names).rename('truth'),
], axis=1)

,poisson,quasi-poisson,truth
Intercept,0.351345,0.351345,0.148997
x1,-0.059991,-0.059991,-0.039704
x2,-0.225998,-0.225998,-0.202066
x3,-0.235400,-0.235400,-0.163432
x4,-0.325585,-0.325585,-0.343295
C(region)[T.b],-0.430820,-0.430820,-0.340730
C(region)[T.c],0.098468,0.098468,0.187761


The `Poisson` and `QuasiPoisson` models estimate the exact same coefficients. This is unsurprising on reflection, they maximize the same likelihood function.

When comparing the estimated coefficients to the truth there is an interesting observation: while the coefficients for the variables are well estimated, the intercept estimate seems far too large. This happens even though the random noise we added to the predictor was mean zero.

See the appendix for an explanation of the intercept bias, and how it comes to be absorbed into the intercept.



Another problem is the standard errors. The Poisson model's assume a dispersion of 1, far below the estimate above, so they're too small, and its resulting intervals are too narrow. 

The coverage check shows how much:

In [14]:
overdispersed_coverage = pd.DataFrame(
    {
        "Poisson": coverage(Poisson(), **overdispersed),
        "QuasiPoisson": coverage(QuasiPoisson(), **overdispersed),
    }
)
overdispersed_coverage.round(3)

,Poisson,QuasiPoisson
Intercept,0.586,0.894
x1,0.534,0.880
x2,0.484,0.870
x3,0.520,0.904
x4,0.542,0.900
C(region)[T.b],0.546,0.892
C(region)[T.c],0.538,0.878


The Poisson model's "95%" intervals cover the truth far less often than 95% of the time. `QuasiPoisson` has the same estimates, but estimates the dispersion, and scales its standard errors up by `√φ̂`. That helps a great deal, but it doesn't fully restore the coverage. The reason is a misspecification of the variance function of the model, see the second appendix for details.

### The bootstrap as a diagnostic

In practice we don't know whether a model's variance assumption holds. The two bootstraps can tell us, because they rely on the model to different degrees:

- The **parametric bootstrap** simulates from the fit model, *including its variance assumption*. For a Poisson model, it simulates Poisson counts, so it reproduces the Poisson model's standard errors, right or wrong.
- The **non-parametric bootstrap** resamples the observed rows, each with its actual response, so it makes no assumption about the variance. It estimates the *robust*, or *sandwich*, standard errors, which are valid whatever the variance function.

So when the two disagree, the model's variance assumption is suspect:

In [15]:
bootstrap = Simulation(poisson_model)
np.random.seed(4)
parametric = coefficient_table(
    bootstrap.parametric_bootstrap(sim.data, n_sim=500, offset=sim.offset)
)
non_parametric = coefficient_table(
    bootstrap.non_parametric_bootstrap(sim.data, n_sim=500, offset=sim.offset)
)
standard_errors = pd.DataFrame(
    {
        "Poisson": poisson_model.coef_standard_error_,
        "parametric bootstrap": parametric.std(),
        "non-parametric bootstrap": non_parametric.std(),
        "QuasiPoisson": quasi_model.coef_standard_error_,
    },
    index=poisson_model.X_names,
)
standard_errors.round(4)

,Poisson,parametric bootstrap,non-parametric bootstrap,QuasiPoisson
Intercept,0.0128,0.0133,0.0402,0.0318
x1,0.0075,0.0072,0.0189,0.0184
x2,0.0073,0.0070,0.0219,0.0180
x3,0.0086,0.0086,0.0254,0.0213
x4,0.0086,0.0084,0.0263,0.0213
C(region)[T.b],0.0194,0.0200,0.0579,0.0479
C(region)[T.c],0.0170,0.0168,0.0525,0.0421


The parametric bootstrap agrees with the Poisson model's standard errors, because it inherits the Poisson assumption. The non-parametric bootstrap's are many times larger: it sees the extra variation in the actual data. That gap is the diagnostic.

They're larger even than `QuasiPoisson`'s, which is consistent with `QuasiPoisson`'s intervals falling short of 95% coverage: the non-parametric bootstrap makes no assumption about the variance's shape, so it captures the extra variation in the high-mean observations that a single dispersion misses.

In [16]:
(standard_errors["non-parametric bootstrap"] / standard_errors["Poisson"]).round(2)

Intercept         3.13
x1                2.54
x2                3.01
x3                2.95
x4                3.05
C(region)[T.b]    2.99
C(region)[T.c]    3.09
dtype: float64

## A caveat: rare categories

The non-parametric bootstrap resamples rows, so a resample can miss a rare category entirely. Then that category's coefficient can't be estimated from the resample, and the bootstrap raises an error, rather than returning misleading results:

In [17]:
rare = make_frame(
    Poisson(),
    n_samples=500,
    n_uncorrelated=2,
    categorical={"plan": [0.6, 0.396, 0.004]},
    rng=np.random.default_rng(5),
)
rare_model = GLM(Poisson()).fit(rare.data, formula=rare.formula)
np.random.seed(6)
try:
    Simulation(rare_model).non_parametric_bootstrap(rare.data, n_sim=200)
except ValueError as error:
    print(f"ValueError: {error}")

ValueError: A bootstrap resample has no observations for design columns ['C(plan)[T.c]'], so their coefficients cannot be estimated. This happens with rare categories; consider merging them.


Options are to merge the rare category into a larger one, or to fit a ridge model, whose penalty keeps every coefficient estimable; the notebook on regularization covers those.

## Appendices

### Appendix: The Intercept Bias
The reason for this bias can be understood with [Jensen's Inequality](https://en.wikipedia.org/wiki/Jensen%27s_inequality), remembering that `log` is a *strictly concave* function.

If we start with our data generating equation:

```
y | X ~ Poisson(λ = exp(X @ β + ϵ))
```

Take conditional expectations:

```
E[y | X, ϵ] = exp(X @ β + ϵ)
    => log(E[y | X, ϵ]) = X @ β + ϵ
    => X @ β = log(E[y | X, ϵ]) - ϵ
```    

Now take the expectation over `ϵ`, which is mean zero, and use Jensen's inequality:

```
X @ β = E[log(E[y | X, ϵ]) | X] 
      < log(E[E[y | X, ϵ] | X])  (Jensen)
      = log(E[y | X])
```

The true coefficients lead to a strictly too small estimate of `log(E[y | X])`. 

Maybe surprisingly, here we can do better and actually compute the amount of bias, which will turn out to be a constant regardless of `X`. To do so, we will introduce conditioning on `ϵ` and apply the tower law for conditional expectations. We compute:

```
E[y | X] = E[E[y | X, ϵ] | X]      (Outside expectation over ϵ)
         = E[exp(X @ β + ϵ) | X]      
         = exp(X @ β) E[exp(ϵ)]    (ϵ is independent of X)
         = exp(X @ β) exp(σ²/2)    (E[lognormal])
```

So the bias is:

```
log(E[y | X]) - X @ β = σ²/2
```

When fitting the model, this constant bias is absorbed into the intercept, which explains what we saw in precise, quantitative terms.

### Appendix: Heteroskedasticity

the `QuasiPoisson` family assumes the variance is proportional to the mean, `Var[y | X] = φ E[y | X]`, with the same φ for every observation. Latent noise doesn't produce that.

Write `λ = exp(X @ β + ϵ)` for the mean *given* the noise, so t
hat `y | X, ϵ ~ Poisson(λ)`, and `μ = E[y | X]` for the mean without it. The law of total variance, introducing and then integrating away the conditioning on `ϵ`, gives:

```
Var[y | X] = E[Var(y | X, ϵ) | X] + Var[E[y | X, ϵ] | X] 
           = E[λ | X] + Var(λ | X)
```

The first term is the mean, `E[λ | X] = μ`, this is more-or-less the tower law calculation in our first appendix:

```
E[λ | X] = E[E[y | X, ϵ] | X]
         = E[y | X]
         = μ
```

The second is the variance of the conditional *mean*, `λ`. For normal `ϵ`, `E[exp(tϵ)] = exp(t²σ² / 2)`, again the same calculation as in the appendix. Applying this with `t = 1` and `t = 2`:

```
Var(λ | X) = exp(2 X @ β) (E[exp(2ϵ)] - E[exp(ϵ)]²)
           = exp(2 X @ β) (exp(2σ²) - exp(σ²))
           = exp(2 X @ β) exp(σ²) (exp(σ²) - 1)
           = (exp(σ²) - 1) μ²
```

using `μ = exp(X @ β) exp(σ² / 2)` from the first appendix. So:

```
Var[y | X] = μ + (exp(σ²) - 1) μ²
```

This can't be written as a fixed multiple of the mean:

```
μ + (exp(σ²) - 1) μ² = (1 + (exp(σ²) - 1) μ) μ
```

The effective dispersion, `1 + (exp(σ²) - 1) μ`, *grows* with the mean: observations with small counts are only slightly overdispersed, while those with large counts are much more so. The observations with large counts also carry the most weight in the fit. So a single φ, which matches the variance's average level but not its shape, leaves the standard errors too small. (We note that: a variance of the mean plus a multiple of its square is the variance function of the negative binomial distribution, the standard model for overdispersed counts.)